In [ ]:
#!pip install transformers datasets accelerate
import transformers
print(transformers.__version__) 

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    pipeline
)
from datasets import load_dataset
from peft import LoraConfig, get_peft_model


In [ ]:
# 3️⃣ Model & Tokenizer paths
model_path = "/workspace/work/Meta-Llama-3-8B"  # Path inside Docker
dataset_path = "/workspace/work/clean_150k.jsonl"    # Your JSONL file
output_dir = "/workspace/work/Meta_Llama_finetuned"   # Save fine-tuned model here


In [ ]:
# Use the correct variable 'model_path' defined in Cell 3
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the model directly to the GPU
model = AutoModelForCausalLM.from_pretrained(
    model_path,                # Fixed variable name
    torch_dtype=torch.float16,
    device_map=None            # We will move it manually to avoid 'meta' device bugs
).to("cuda")

# This is crucial for saving memory during training
model.gradient_checkpointing_enable()

# Verify the device - it should say 'cuda:0'
print(f"Model is on device: {next(model.parameters()).device}")

In [ ]:
from transformers import BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=quant_config, # This shrinks the model from 15GB to ~5GB
    device_map={"": 0}
)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Setup the LoRA config
config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], # Targets specific layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 2. Wrap the model
model = get_peft_model(model, config)

# 3. Important: Verify only ~1-2% of params are trainable
model.print_trainable_parameters()

In [ ]:
dataset = load_dataset("json", data_files=dataset_path)["train"]
dataset = dataset.train_test_split(train_size=0.7)

# Tokenization function
def tokenize_fn(examples):
    input_ids_list = []
    attention_masks = []
    labels_list = []

    for instruction, response in zip(examples["instruction"], examples["response"]):
        prompt = f"Instruction: {instruction}\nResponse:"
        full_text = f"{prompt} {response}{tokenizer.eos_token}"

        # Tokenize full text
        tokenized = tokenizer(
            full_text,
            truncation=True,
            padding="max_length",   # important for stable masking
            max_length=512
        )

        # Tokenize prompt only (to know where to mask)
        prompt_ids = tokenizer(
            prompt,
            truncation=True,
            max_length=512
        )["input_ids"]

        labels = tokenized["input_ids"].copy()

        # 🔴 MASK instruction part
        labels[:len(prompt_ids)] = [-100] * len(prompt_ids)

        input_ids_list.append(tokenized["input_ids"])
        attention_masks.append(tokenized["attention_mask"])
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_masks,
        "labels": labels_list,
    }

tokenized_dataset = dataset.map(tokenize_fn, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["instruction", "response"])
tokenized_dataset.set_format("torch")

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # causal LM
)

In [ ]:
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    num_train_epochs=0.5,
    logging_steps=1,
    save_steps=500,
    fp16=True,  # use if GPU supports, optional
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,       # RE-ENABLE THIS
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator
)

In [ ]:
trainer.train()
trainer.save_model(output_dir)

In [ ]:
pipe = pipeline(
    "text-generation",
    model=output_dir,
    tokenizer=tokenizer
)

In [ ]:
prompt = "Instruction: How can I make focus follow the mouse cursor?\nResponse:"
output = pipe(prompt, max_new_tokens=256, temperature=0.7, top_p=0.9, do_sample=True)
print(output[0]["generated_text"])